# 15 — CorrDiff two-stage downscaling

Notebook 09 trained a *single* conditional denoiser to jump straight from the blurred
observation to the sharp field. CorrDiff (and StormCast) split that job in two:

1. a deterministic **regression** stage learns the conditional mean — the easy, smooth part,
2. a diffusion stage denoises only the **residual** `target − regression(condition)` — the
   sharp, genuinely stochastic remainder.

The ensemble mean then comes out sharper and the spread better calibrated, because the
denoiser no longer wastes capacity re-predicting the mean. Here:

- the same analytic Gaussian-mixture pairs as notebook 09 (a blurred Gaussian is a wider
  Gaussian, so ground truth is exact),
- `TrainCorrDiffPair` runs stage 1 (`RegressionLoss` on a `CorrDiffRegressionUNet`) and
  stage 2 (`ResidualLoss` with the frozen stage-1 net) in one call,
- `DiffusionInferenceProcess` deploys the pair through its `"regression_settings"` block:
  the regression mean is added to the generated ensemble before the mean/std are taken.

One sizing rule to know up front: `ResidualLoss` drives the denoiser with positional-embedding
kwargs, so the residual-stage denoiser must wrap `SongUNetPosEmbd` — whose `N_grid_channels`
(default 4) count toward `img_in_channels`, exactly as upstream CorrDiff configs size it.


In [1]:
import math
from pathlib import Path

import numpy
import torch

import KratosMultiphysics as Kratos
from KratosMultiphysics.PhysicsNeMoApplication import diffusion_inference_process
from KratosMultiphysics.PhysicsNeMoApplication import diffusion_utils
from KratosMultiphysics.PhysicsNeMoApplication import grid_dataset_export_process
from KratosMultiphysics.PhysicsNeMoApplication import training_utils
from KratosMultiphysics.PhysicsNeMoApplication.torch_dataset import CreateGridPairDataset

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

_HEX_TO_TETS = ((0, 1, 2, 6), (0, 2, 3, 6), (0, 3, 7, 6), (0, 7, 4, 6), (0, 4, 5, 6), (0, 5, 1, 6))

def create_tet_cube(model, name, divisions, variables):
    model_part = model.CreateModelPart(name)
    for variable in variables:
        model_part.AddNodalSolutionStepVariable(variable)
    props = model_part.CreateNewProperties(1)
    n = divisions + 1
    for i in range(n):
        for j in range(n):
            for k in range(n):
                model_part.CreateNewNode(i * n * n + j * n + k + 1,
                                         i / divisions, j / divisions, k / divisions)
    nid = lambda i, j, k: i * n * n + j * n + k + 1
    element_id = 0
    for i in range(divisions):
        for j in range(divisions):
            for k in range(divisions):
                corners = [nid(i, j, k), nid(i + 1, j, k), nid(i + 1, j + 1, k), nid(i, j + 1, k),
                           nid(i, j, k + 1), nid(i + 1, j, k + 1), nid(i + 1, j + 1, k + 1), nid(i, j + 1, k + 1)]
                for tet in _HEX_TO_TETS:
                    element_id += 1
                    model_part.CreateNewElement("Element3D4N", element_id, [corners[c] for c in tet], props)
    return model_part

BLUR = 0.012  # variance added by the "coarse observation" operator

def mixture(x, y, params, blur=0.0):
    """Two-Gaussian mixture; blur adds variance (exact Gaussian smoothing)."""
    value = 0.0
    for (cx, cy, var, amplitude) in params:
        total_var = var + blur
        value += amplitude * (var / total_var) * math.exp(
            -((x - cx) ** 2 + (y - cy) ** 2) / (2.0 * total_var))
    return value

def draw_params(rng):
    return [(rng.uniform(0.25, 0.75), rng.uniform(0.25, 0.75),
             rng.uniform(0.004, 0.010), rng.uniform(0.6, 1.0)) for _ in range(2)]

model = Kratos.Model()
sharp_part = create_tet_cube(model, "Sharp", 6, (Kratos.TEMPERATURE,))
blurred_part = create_tet_cube(model, "Blurred", 6, (Kratos.TEMPERATURE, Kratos.NODAL_ERROR))
print("two parts on the same unit cube:", sharp_part.NumberOfNodes(), "nodes each")


two parts on the same unit cube: 343 nodes each


## Matched (condition, target) pairs

Identical to notebook 09: two `GridDatasetExportProcess` instances write step-matched
series that `CreateGridPairDataset` zips into training pairs. A static conditioning
channel (topography, masks, material maps) would simply be one more entry in
`list_of_fields`/`input_fields` — the plumbing never asks what the channels mean.


In [2]:
def make_export(part_name, path):
    settings = Kratos.Parameters("""{
        "Parameters": {
            "model_part_name" : "%s",
            "list_of_fields"  : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_historical" } ],
            "grid_shape"      : [16, 16, 2],
            "output_path"     : "%s"
        }
    }""" % (part_name, path))
    process = grid_dataset_export_process.Factory(settings, model)
    process.ExecuteInitialize()
    return process

export_sharp = make_export("Sharp", "output/corrdiff_sharp")
export_blurred = make_export("Blurred", "output/corrdiff_blurred")

rng = numpy.random.default_rng(0)
N_SAMPLES = 48
for step in range(1, N_SAMPLES + 1):
    params = draw_params(rng)
    for node in sharp_part.Nodes:
        node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, params))
    for node in blurred_part.Nodes:
        node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, params, blur=BLUR))
    for part, process in ((sharp_part, export_sharp), (blurred_part, export_blurred)):
        part.ProcessInfo[Kratos.STEP] = step
        part.ProcessInfo[Kratos.TIME] = float(step)
        process.ExecuteFinalizeSolutionStep()

pairs = CreateGridPairDataset("output/corrdiff_blurred", "output/corrdiff_sharp", squeeze_axis=2)
print(len(pairs), "matched (blurred, sharp) pairs")


48 matched (blurred, sharp) pairs


## Both stages in one call

Stage 1 is a `CorrDiffRegressionUNet` (the `net(zeros, img_lr)` interface) trained with
upstream's `RegressionLoss` — plain MSE against the condition, i.e. the conditional mean.
Stage 2 is the usual EDM-preconditioned denoiser, but sized for `SongUNetPosEmbd`
(`img_in_channels = 1 condition channel + 4 positional grid channels`) and trained with
`ResidualLoss(regression_net=...)`; `TrainCorrDiffPair` freezes stage 1 explicitly
(`.eval()` + `requires_grad_(False)`) and selects the residual loss's `P_mean = 0.0`
default (residuals are centered — the `edm_sr` default of −1.2 would be wrong here).


In [3]:
from physicsnemo.diffusion.preconditioners import EDMPrecondSuperResolution
from physicsnemo.models.diffusion_unets import CorrDiffRegressionUNet

torch.manual_seed(0)
regression = CorrDiffRegressionUNet(
    img_resolution=16, img_in_channels=1, img_out_channels=1,
    model_type="SongUNet", model_channels=16, channel_mult=[1, 2],
    num_blocks=1, attn_resolutions=[])

torch.manual_seed(1)
denoiser = EDMPrecondSuperResolution(
    img_resolution=16, img_in_channels=1 + 4, img_out_channels=1,
    model_type="SongUNetPosEmbd", model_channels=16, channel_mult=[1, 2],
    num_blocks=1, attn_resolutions=[])

regression_history, diffusion_history = diffusion_utils.TrainCorrDiffPair(
    regression, denoiser, pairs, Kratos.Parameters("""{
        "epochs"                   : 120,
        "regression_epochs"        : 200,
        "batch_size"               : 8,
        "learning_rate"            : 2e-4,
        "regression_learning_rate" : 1e-3,
        "echo_interval"            : 40,
        "seed"                     : 0
    }"""))
print(f"regression loss: {regression_history[0]:.3e} -> {regression_history[-1]:.3e}")
print(f"residual   loss: {diffusion_history[0]:.3e} -> {diffusion_history[-1]:.3e}")

card = {
    "input_fields":  [{"variable_name": "TEMPERATURE", "data_location": "node_historical"}],
    "output_fields": [{"variable_name": "TEMPERATURE", "data_location": "node_non_historical"}],
}
training_utils.SaveTrainedModel(regression, OUTPUT / "corrdiff_regression.mdlus", card=card)
training_utils.SaveTrainedModel(denoiser, OUTPUT / "corrdiff_residual.mdlus", card=card)


/home/vicente/.local/lib/python3.12/site-packages/physicsnemo/models/diffusion_unets/song_unet.py:962: FutureWarning: gridtype="sinusoidal" uses a legacy frequency-band formula that does not produce exact octave doublings. Use "sinusoidal_octave" for new models. Only use "sinusoidal" when loading pre-trained checkpoints.
  "pos_embd", self._get_positional_embedding().float(), persistent=False
regression loss: 1.108e-02 -> 6.811e-05
residual   loss: 3.634e-01 -> 9.590e-03


## Deployment: `regression_settings` on the inference process

The only change against notebook 09's deployment is the `"regression_settings"` block —
`model_settings`-shaped, loaded through the same registry with the same model-card checks.
When present, the regression mean of the sampled condition is added to every ensemble
member **before** the mean/std reductions: the written mean shifts by the deterministic
stage, while the uncertainty stays the residual ensemble's spread — exactly CorrDiff
inference.


In [4]:
held_out = draw_params(numpy.random.default_rng(1234))
for node in blurred_part.Nodes:
    node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, held_out, blur=BLUR))

process = diffusion_inference_process.Factory(Kratos.Parameters("""{
    "Parameters": {
        "model_part_name"     : "Blurred",
        "model_settings"      : {
            "checkpoint_file" : "output/corrdiff_residual.mdlus",
            "checkpoint_type" : "physicsnemo"
        },
        "regression_settings" : {
            "checkpoint_file" : "output/corrdiff_regression.mdlus",
            "checkpoint_type" : "physicsnemo"
        },
        "input_fields"        : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_historical" } ],
        "output_fields"       : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_non_historical" } ],
        "uncertainty_fields"  : [ { "variable_name" : "NODAL_ERROR", "data_location" : "node_non_historical" } ],
        "grid_shape"          : [16, 16, 2],
        "squeeze_axis"        : 2,
        "sampler_settings"    : { "num_samples" : 8, "num_steps" : 12, "seed" : 0 }
    }
}"""), model)

blurred_part.ProcessInfo[Kratos.STEP] = 1
process.ExecuteFinalizeSolutionStep()

truth = numpy.array([mixture(node.X, node.Y, held_out) for node in blurred_part.Nodes])
condition = numpy.array([node.GetSolutionStepValue(Kratos.TEMPERATURE) for node in blurred_part.Nodes])
predicted = numpy.array([node.GetValue(Kratos.TEMPERATURE) for node in blurred_part.Nodes])
spread = numpy.array([node.GetValue(Kratos.NODAL_ERROR) for node in blurred_part.Nodes])

rmse = lambda a: float(numpy.sqrt(numpy.mean((a - truth) ** 2)))
print(f"rmse(blurred condition vs truth)  = {rmse(condition):.4f}")
print(f"rmse(two-stage ensemble vs truth) = {rmse(predicted):.4f}")
print(f"mean uncertainty (residual std)   = {float(spread.mean()):.4f}")


/home/vicente/.local/lib/python3.12/site-packages/physicsnemo/models/diffusion_unets/song_unet.py:962: FutureWarning: gridtype="sinusoidal" uses a legacy frequency-band formula that does not produce exact octave doublings. Use "sinusoidal_octave" for new models. Only use "sinusoidal" when loading pre-trained checkpoints.
  "pos_embd", self._get_positional_embedding().float(), persistent=False
rmse(blurred condition vs truth)  = 0.0542
rmse(two-stage ensemble vs truth) = 0.0256
mean uncertainty (residual std)   = 0.0363


## How the stages divide the work

The regression stage alone is a strong deterministic baseline; the residual ensemble adds
the stochastic sharpening plus the uncertainty band. Comparing the two on the same held-out
field shows the split:


In [5]:
from KratosMultiphysics.PhysicsNeMoApplication import grid_bridge

for node in sharp_part.Nodes:
    node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, held_out))

def grid_of(part):
    grid, _ = grid_bridge.SampleFieldsOnGrid(
        part, [("TEMPERATURE", "node_historical")], (16, 16, 2))
    return grid.mean(axis=3)  # thin-axis idiom: collapse the size-2 axis -> (C, 16, 16)

truth_grid = grid_of(sharp_part)
condition_grid = grid_of(blurred_part)
regression_only = diffusion_utils.RunRegressionMean(regression, condition_grid)

grid_rmse = lambda a: float(numpy.sqrt(numpy.mean((a - truth_grid) ** 2)))
print(f"grid rmse, blurred condition   = {grid_rmse(condition_grid):.4f}")
print(f"grid rmse, regression alone    = {grid_rmse(regression_only):.4f}")
print("the residual ensemble adds the stochastic sharpening on top,",
      f"and its spread ({float(spread.mean()):.4f}) is the calibrated uncertainty")


grid rmse, blurred condition   = 0.0428
grid rmse, regression alone    = 0.0151
the residual ensemble adds the stochastic sharpening on top, and its spread (0.0363) is the calibrated uncertainty


## Variations

- **Wind/dam downscaling**: condition = the coarse RANS or seepage solve plus a *static*
  topography channel (one more `input_fields` entry that never changes), target = the fine
  field — the CorrDiff setting for `WindEngineeringApplication`/`DamApplication` cases.
- **FWI-style subsurface inversion**: condition = sparse borehole observations plus a binary
  observation-mask channel, target = the subsurface property grid; `GenerateEnsemble`'s
  per-node standard deviation is the inversion uncertainty. The layered-earth recipe is
  pinned by `tests/test_corrdiff_recipe.py::TestFwiInversionRecipe`.
- **Sharper ensembles**: raise `num_steps`/`num_samples`; more regression epochs move more
  of the error into the deterministic stage and shrink what the denoiser must model.
